In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [ ]:
# ============================================================
# Import Libraries
# ============================================================

from pathlib import Path
import string

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import (
    ENGLISH_STOP_WORDS,
    TfidfVectorizer
)

from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# ============================================================
# Dataset Paths
# ============================================================

PROJECT_DIR = Path.cwd()

KAGGLE_DATA_DIR = Path(
    "/kaggle/input/competitions/smart-mcq-solver-challenge"
)

if KAGGLE_DATA_DIR.exists():
    DATA_DIR = KAGGLE_DATA_DIR
    OUTPUT_DIR = Path("/kaggle/working")
else:
    DATA_DIR = PROJECT_DIR / "data"
    OUTPUT_DIR = PROJECT_DIR / "outputs"

OPTION_LABELS = np.array(list("ABCDE"))

In [ ]:
# ============================================================
# Load Dataset
# ============================================================

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

print(train.shape)
print(test.shape)

train.head()

In [ ]:
# ============================================================
# Helper Functions
# ============================================================

def clean_prompt(text):
    return text.lower().translate(
        str.maketrans("", "", string.punctuation)
    )


def combined_text(df):
    columns = ["prompt", *OPTION_LABELS]
    return df[columns].fillna("").agg(" ".join, axis=1)


def option_similarities(vectorizer, df):

    similarities = []

    for _, row in df.iterrows():

        prompt_vec = vectorizer.transform([row["prompt"]])

        scores = [
            cosine_similarity(
                prompt_vec,
                vectorizer.transform([row[option]])
            )[0, 0]

            for option in OPTION_LABELS
        ]

        similarities.append(scores)

    return np.asarray(similarities)


def rank_options(similarities):

    return [
        [
            label
            for label, _
            in sorted(
                zip(OPTION_LABELS, row),
                key=lambda x: x[1],
                reverse=True
            )
        ]

        for row in similarities
    ]


def map_at_3(y_true, preds):

    scores = []

    for truth, pred in zip(y_true, preds):

        if truth in pred:
            scores.append(1 / (pred.index(truth) + 1))
        else:
            scores.append(0)

    return np.mean(scores)

In [ ]:
# ============================================================
# Exploratory Data Analysis
# ============================================================

answer_counts = train["answer"].value_counts().sort_index()

display(answer_counts)

print("Most + Least Frequent:",
      answer_counts.max() + answer_counts.min())

In [ ]:
# ============================================================
# Text Cleaning
# ============================================================

cleaned_prompts = train["prompt"].map(clean_prompt)

vocab = set(
    " ".join(cleaned_prompts).split()
)

print("Vocabulary Size:", len(vocab))

In [ ]:
# ============================================================
# Stopword Removal Example
# ============================================================

row_prompt = cleaned_prompts.loc[
    train["id"] == 1
].iloc[0]

filtered_words = [

    word

    for word in row_prompt.split()

    if word not in ENGLISH_STOP_WORDS

]

print(filtered_words)
print(len(filtered_words))

In [ ]:
# ============================================================
# TF-IDF Baseline
# ============================================================

vectorizer = TfidfVectorizer(
    stop_words="english"
)

vectorizer.fit(
    combined_text(train)
)

print(
    "Vocabulary Size:",
    len(vectorizer.get_feature_names_out())
)

In [ ]:
# ============================================================
# Cosine Similarity
# ============================================================

train_similarity = option_similarities(
    vectorizer,
    train
)

train_rankings = rank_options(
    train_similarity
)

top1 = np.array([
    row[0]
    for row in train_rankings
])

print(
    "Row 1 Prompt vs Option A:",
    train_similarity[0,0]
)

print(
    "Top-1 Accuracy:",
    (top1 == train.answer).mean()
)

print(
    "MAP@3:",
    map_at_3(train.answer, train_rankings)
)

In [ ]:
# ============================================================
# Majority Baseline
# ============================================================

majority = train.answer.value_counts().index[:3].tolist()

majority_preds = [
    majority
] * len(train)

print(majority)

print(
    map_at_3(
        train.answer,
        majority_preds
    )
)

In [ ]:
# ============================================================
# Generate Submission
# ============================================================

test_similarity = option_similarities(
    vectorizer,
    test
)

test_rankings = rank_options(
    test_similarity
)

submission = pd.DataFrame({

    "ID": test.id,

    "Prediction": [
        " ".join(row[:3])
        for row in test_rankings
    ]

})

OUTPUT_DIR.mkdir(exist_ok=True)

submission.to_csv(
    OUTPUT_DIR / "submission.csv",
    index=False
)

submission.head()

# Milestone 2

In [5]:
"""
Milestone 2 solution script.
Run this in a Kaggle Notebook or Google Colab (needs internet access to
download model weights from Hugging Face Hub). Place train.csv in the
same directory, or point data_files to its path.

pip install -q transformers datasets sentence-transformers torch
"""

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, pipeline
from sentence_transformers import SentenceTransformer, util

# ---------------------------------------------------------------------
# Q1: combined_text length at index 51
# ---------------------------------------------------------------------
ds = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")["train"]
ds = ds.map(lambda x: {"combined_text": x["prompt"] + " " + x["A"]})
print("Q1 combined_text length @51:", len(ds[51]["combined_text"]))
# -> 614 (verified directly from the CSV)

# ---------------------------------------------------------------------
# Q2 & Q3: tokenizer vocab size and [SEP] id
# ---------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print("Q2 vocab_size:", tokenizer.vocab_size)          # 30522
print("Q3 [SEP] id:", tokenizer.sep_token_id)           # 102

# ---------------------------------------------------------------------
# Q4: tokenize entire prompt column
# ---------------------------------------------------------------------
prompt_list = [str(p) for p in ds["prompt"]]  # force plain python str
encoded = tokenizer(
    prompt_list, padding="max_length", truncation=True,
    max_length=128, return_tensors="pt"
)
print("Q4 input_ids shape:", encoded["input_ids"].shape)  # [2000, 128]

# ---------------------------------------------------------------------
# Q5: attention head dimensionality
# ---------------------------------------------------------------------
print("Q5 head dim:", 768 // 12)  # 64

# ---------------------------------------------------------------------
# Q6 & Q7: last_hidden_state shape + CLS vector sum
# ---------------------------------------------------------------------
model = AutoModel.from_pretrained("bert-base-uncased")
model.eval()

row0_prompt = ds[0]["prompt"]
inputs = tokenizer(row0_prompt, return_tensors="pt")  # default settings
with torch.no_grad():
    out = model(**inputs)

print("Q6 last_hidden_state shape:", out.last_hidden_state.shape)

cls_vec = out.last_hidden_state[0, 0, :]  # [CLS] is token index 0
cls_sum_first5 = cls_vec[:5].sum().item()
print("Q7 sum of first 5 CLS values:", round(cls_sum_first5, 4))

# ---------------------------------------------------------------------
# Q8: attention weight from [CLS] to "fusion"
# ---------------------------------------------------------------------
attn_model = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
attn_model.eval()

text = "Light-ion fusion is a technique."
attn_inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    attn_out = attn_model(**attn_inputs)

# find token index for "fusion"
tokens = tokenizer.convert_ids_to_tokens(attn_inputs["input_ids"][0])
print("tokens:", list(enumerate(tokens)))
fusion_idx = tokens.index("fusion")

last_layer_attn = attn_out.attentions[-1]        # (batch, heads, seq, seq)
cls_to_fusion = last_layer_attn[0, 0, 0, fusion_idx].item()
print("Q8 [CLS]->fusion attention weight:", round(cls_to_fusion, 4))

# ---------------------------------------------------------------------
# Q9: MiniLM cosine similarity, prompt vs Option B, row 0
# ---------------------------------------------------------------------
minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
emb_prompt = minilm.encode(ds[0]["prompt"])
emb_b = minilm.encode(ds[0]["B"])
sim = util.cos_sim(emb_prompt, emb_b).item()
print("Q9 cosine similarity prompt vs B (row 0):", round(sim, 4))

# ---------------------------------------------------------------------
# Q10: MAP@3 — TF-IDF pipeline vs MiniLM pipeline
# ---------------------------------------------------------------------
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

options_cols = ["A", "B", "C", "D", "E"]

def map_at_3(top3_lists, answers):
    scores = []
    for top3, ans in zip(top3_lists, answers):
        if ans not in top3:
            scores.append(0.0)
        else:
            rank = top3.index(ans) + 1
            scores.append(1.0 / rank)
    return sum(scores) / len(scores)

# --- Pipeline 1: TF-IDF ---
tfidf_top3 = []
for row in ds:
    corpus = [row["prompt"]] + [row[c] for c in options_cols]
    vec = TfidfVectorizer().fit_transform(corpus)
    sims = cosine_similarity(vec[0:1], vec[1:]).flatten()
    order = np.argsort(-sims)
    top3 = [options_cols[i] for i in order[:3]]
    tfidf_top3.append(top3)

# --- Pipeline 2: MiniLM ---
minilm_top3 = []
for row in ds:
    prompt_emb = minilm.encode(row["prompt"])
    opt_embs = minilm.encode([row[c] for c in options_cols])
    sims = util.cos_sim(prompt_emb, opt_embs).flatten().numpy()
    order = np.argsort(-sims)
    top3 = [options_cols[i] for i in order[:3]]
    minilm_top3.append(top3)

answers = ds["answer"]
map3_minilm = map_at_3(minilm_top3, answers)
print("Q10a MAP@3 (MiniLM):", round(map3_minilm, 4))

count_gained = 0
for t3_tfidf, t3_minilm, ans in zip(tfidf_top3, minilm_top3, answers):
    if ans not in t3_tfidf and ans in t3_minilm:
        count_gained += 1
print("Q10b count improved by MiniLM over TF-IDF:", count_gained)

# ---------------------------------------------------------------------
# Q11 & Q12: zero-shot classification
# ---------------------------------------------------------------------
zsc = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

row1 = ds[1]
candidates = [row1["A"], row1["B"], row1["C"]]

result_softmax = zsc(row1["prompt"], candidate_labels=candidates)
top_score = result_softmax["scores"][0]
print("Q11 top-ranked score (softmax):", round(top_score, 4))

result_sigmoid = zsc(row1["prompt"], candidate_labels=candidates, multi_label=True)
sum_softmax = sum(result_softmax["scores"])
sum_sigmoid = sum(result_sigmoid["scores"])
print("Q12 |sum(softmax) - sum(sigmoid)|:", round(abs(sum_softmax - sum_sigmoid), 4))

# ---------------------------------------------------------------------
# Q13: Flan-T5-small generative QA
# ---------------------------------------------------------------------
from transformers import AutoModelForSeq2SeqLM

t5_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
t5_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")
t5_model.eval()

row0 = ds[0]
prompt_str = (
    f"Question: {row0['prompt']}. Is the correct answer "
    f"A: {row0['A']} or B: {row0['B']}? Answer with just the letter A or B."
)
t5_inputs = t5_tokenizer(prompt_str, return_tensors="pt")
with torch.no_grad():
    t5_out_ids = t5_model.generate(**t5_inputs, max_new_tokens=5)
t5_output_text = t5_tokenizer.decode(t5_out_ids[0], skip_special_tokens=True)
print("Q13 model output:", t5_output_text)

Q1 combined_text length @51: 614
Q2 vocab_size: 30522
Q3 [SEP] id: 102
Q4 input_ids shape: torch.Size([2000, 128])
Q5 head dim: 64


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q6 last_hidden_state shape: torch.Size([1, 31, 768])
Q7 sum of first 5 CLS values: -1.2001


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Q13 model output: B
